#Q1

In [1]:
scores_csv = """이름,학번,중간,기말,과제
김언어,2026-10000,85,92,90
이국문,2026-12345,78,88,85
박영문,2026-13579,95,90,100
최역사,2025-11111,,82,88
"""
with open("scores.csv", "w", encoding="utf-8") as f:
    f.write(scores_csv)

In [2]:
import csv
import json
import logging

In [6]:
logging.basicConfig(level=logging.INFO, force=True)

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        logging.warning(f"WARNING: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"ERROR: {csv_path}")
        return 0

    results = []
    for row in rows:
        name   = row["이름"]
        sid    = row["학번"]
        mid_s  = row["중간"].strip()
        fin_s  = row["기말"].strip()
        hw_s   = row["과제"].strip()

        mid  = int(mid_s)  if mid_s  else None
        fin  = int(fin_s)  if fin_s  else None
        hw   = int(hw_s)   if hw_s   else None

        if None in (mid, fin, hw):
            avg   = None
            grade = None
            logging.info(f"{name}: 평균 None, 등급 None (결측값 있음)")
        else:
            avg = mid * 0.3 + fin * 0.5 + hw * 0.2
            if   avg >= 90: grade = "A"
            elif avg >= 80: grade = "B"
            elif avg >= 70: grade = "C"
            else:           grade = "F"
            logging.info(f"{name}: 평균 {avg}, 등급 {grade}")

        results.append({
            "이름": name,
            "학번": sid,
            "점수": {"중간": mid, "기말": fin, "과제": hw},
            "평균": avg,
            "등급": grade,
        })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return len(results)

In [7]:
make_report("scores.csv", "report.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None (결측값 있음)


4

In [8]:
with open("report.json", "r", encoding="utf-8") as f:
    print(f.read())

[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]


결측값 처리는 점수 세 칸 중 하나라도 빈 문자열이면 None으로 통일했다. 인코딩은 한글 텍스트의 표준인 utf-8을 명시했다. 예외는 FileNotFoundError와 UnicodeDecodeError를 통해 버그가 숨겨지지 않도록 했다.

#Q2

In [9]:
#(a)
class InvalidJamoError(ValueError):
    pass

#(b)
def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {type(c)}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {repr(c)}")
    cp = ord(c)
    if 0x3131 <= cp <= 0x314E:
        return "자음"
    if 0x314F <= cp <= 0x3163:
        return "모음"
    raise InvalidJamoError(f"한글 JAMO가 아닙니다: {repr(c)}")

#(c)
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(f"{item!r} -> {result}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ' -> 자음
'ㅏ' -> 모음
'ㄲ' -> 자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이 아닙니다: 'AB'
[TypeError] str 타입이 아닙니다: <class 'int'>
'ㅎ' -> 자음
'ㅣ' -> 모음
[ValueError] 길이가 1이 아닙니다: ''


(a) 이 오류가 잘못된 값이 전달되어 생겼기 때문이다. 그리고 except에서 InvalidJamoError를 ValueError보다 앞에 두어야 자식 클래스가 부모보다 먼저 잡히기 때문이다.